In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import *
from numpy import newaxis

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 15)

In [2]:
df = pd.read_csv('data.csv')

In [3]:
def _standardize_datetime_str(dt_str):
    """Helper: Convert datetime string to YYYY-MM-DD HH:MM:SS format."""
    try:
        if len(dt_str.split(':')) == 3:
            dt = pd.to_datetime(dt_str, format='%Y-%m-%d %H:%M:%S')
        else:
            dt = pd.to_datetime(dt_str, format='%Y-%m-%d %H:%M')
        return dt.strftime('%Y-%m-%d %H:%M:%S')
    except:
        return pd.to_datetime(dt_str).strftime('%Y-%m-%d %H:%M:%S')


def standardize_datetime_column(df, datetime_col='DateTime', verbose=True):
    """
    Pipeline Step 1: Standardize DateTime format to YYYY-MM-DD HH:MM:SS.
    
    Usage with .pipe():
        df.pipe(standardize_datetime_column)
    """
    if verbose:
        print("="*60)
        print("[1/3] Standardizing DateTime column")
        print("="*60)
    
    return (df
            .assign(**{datetime_col: lambda x: x[datetime_col].apply(_standardize_datetime_str)})
            .pipe(lambda df: df if not verbose else 
                  (print(f"✓ Standardized {len(df):,} rows") or df))
           )


def remove_duplicate_observations(df, product_col='Product_Number', datetime_col='DateTime', verbose=True):
    """
    Pipeline Step 2: Remove duplicates, keeping latest observation per Product-Date.
    
    For each Product on the same date, keeps only the latest time.
    
    Usage with .pipe():
        df.pipe(remove_duplicate_observations)
    """
    if verbose:
        print("\n" + "="*60)
        print("[2/3] Removing duplicate observations")
        print("="*60)
        rows_before = len(df)
    
    result = (df
              .assign(_datetime_temp=lambda x: pd.to_datetime(x[datetime_col]))
              .assign(_date_temp=lambda x: x['_datetime_temp'].dt.date)
              .sort_values([product_col, '_date_temp', '_datetime_temp'])
              .drop_duplicates(subset=[product_col, '_date_temp'], keep='last')
              .drop(columns=['_datetime_temp', '_date_temp'])
              .reset_index(drop=True)
             )
    
    if verbose:
        rows_removed = rows_before - len(result)
        print(f"✓ Before: {rows_before:,} rows")
        print(f"✓ After:  {len(result):,} rows")
        print(f"✓ Removed: {rows_removed:,} rows ({100*rows_removed/rows_before:.2f}%)")
    
    return result


def filter_products_by_observation_count(df, product_col='Product_Number', required_count=95, verbose=True):
    """
    Pipeline Step 3: Filter products with exact required observation count.
    
    Removes products that don't have exactly the required number of observations.
    """
    if verbose:
        print("\n" + "="*60)
        print(f"[3/3] Filtering products with {required_count} observations")
        print("="*60)
        products_before = df[product_col].nunique()
        rows_before = len(df)
    
    # Count observations per product
    product_counts = df.groupby(product_col).size()
    
    # Get products with required count
    valid_products = product_counts[product_counts == required_count].index
    
    # Filter dataframe
    result = (df
              .loc[lambda x: x[product_col].isin(valid_products)]
              .reset_index(drop=True)
             )
    
    if verbose:
        products_removed = products_before - result[product_col].nunique()
        rows_removed = rows_before - len(result)
        
        print(f"✓ Products before: {products_before:,}")
        print(f"✓ Products after:  {result[product_col].nunique():,}")
        print(f"✓ Products removed: {products_removed:,}")
        print(f"✓ Rows removed: {rows_removed:,}")
        
        # Show distribution of observation counts
        invalid_counts = product_counts[product_counts != required_count]
        if len(invalid_counts) > 0:
            print(f"\n✓ Removed products had observation counts:")
            for count, num_products in invalid_counts.value_counts().sort_index().items():
                print(f"  - {num_products} product(s) with {count} observations")
    
    return result

In [4]:
# ============================================================
# Apply Preprocessing Pipeline
# ============================================================

df = (df
      .pipe(standardize_datetime_column, datetime_col='DateTime', verbose=True)
      .pipe(remove_duplicate_observations, product_col='Product_Number', datetime_col='DateTime', verbose=True)
      .pipe(filter_products_by_observation_count, product_col='Product_Number', required_count=95, verbose=True)
     )

print("\n" + "="*60)
print("✓ Preprocessing Pipeline Complete!")
print("="*60)


[1/3] Standardizing DateTime column
✓ Standardized 34,617 rows

[2/3] Removing duplicate observations
✓ Before: 34,617 rows
✓ After:  10,624 rows
✓ Removed: 23,993 rows (69.31%)

[3/3] Filtering products with 95 observations
✓ Products before: 117
✓ Products after:  102
✓ Products removed: 15
✓ Rows removed: 934

✓ Removed products had observation counts:
  - 1 product(s) with 28 observations
  - 1 product(s) with 30 observations
  - 1 product(s) with 39 observations
  - 1 product(s) with 45 observations
  - 1 product(s) with 46 observations
  - 2 product(s) with 64 observations
  - 1 product(s) with 67 observations
  - 1 product(s) with 69 observations
  - 3 product(s) with 75 observations
  - 1 product(s) with 79 observations
  - 2 product(s) with 89 observations

✓ Preprocessing Pipeline Complete!


In [ ]:
df